# Half-life revision workflow

This notebook is the interactive entry point for the revision models you trained for the resubmission.

- Use the `parnet_clean` kernel for this notebook.
- Use `tfmodisco` only for the final motif/report shell step.
- The main control is `RUN_NAME`, which maps to one of your original model definitions.


In [38]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd

REPO_ROOT = Path.cwd()
PIPELINE_PATH = REPO_ROOT / "resubmission" / "scripts" / "hl_revision_pipeline.py"

spec = importlib.util.spec_from_file_location("hl_revision_pipeline", PIPELINE_PATH)
hlrev = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(hlrev)

PARNET_PYTHON = Path("/lustre/groups/crna01/projects/collabs/shared/cenvs/parnet_clean/bin/python")
TFMODISCO_PYTHON = Path("/lustre/groups/crna01/projects/collabs/shared/cenvs/tfmodisco/bin/python")
print(f"Kernel python: {sys.executable}")
print(f"Expected parnet_clean python: {PARNET_PYTHON}")
print(f"tfmodisco python: {TFMODISCO_PYTHON}")


Kernel python: /ictstr01/groups/crna01/projects/collabs/shared/cenvs/parnet_clean/bin/python3.10
Expected parnet_clean python: /lustre/groups/crna01/projects/collabs/shared/cenvs/parnet_clean/bin/python
tfmodisco python: /lustre/groups/crna01/projects/collabs/shared/cenvs/tfmodisco/bin/python


In [39]:
from lets_plot import *
LetsPlot.setup_html()
base_size = 14
theme_settings = theme(
    axis_text=element_text(size=base_size),
    axis_title=element_text(size=base_size * 1.2),
    legend_title=element_text(size=base_size * 1.2),
    legend_text=element_text(size=base_size * 0.9),
    plot_title=element_text(size=base_size * 1.4),
    axis_ticks_y=element_line(color='black', size=0.5),
    axis_line_y=element_line(color='black', size=0.5)
)


def plot_trimmed_borders(col, perc_to_excl, data_tab):
    valid = data_tab[col].dropna().sort_values()
    n = len(valid)
    n_each_tail = int((n - round(n * perc_to_excl, 0)) / 2)

    low_cut = valid.iloc[:n_each_tail].max()
    high_cut = valid.iloc[-n_each_tail:].min()

    print(f"Bottom upper {perc_to_excl*100:.1f}% border: {low_cut:.6f}")
    print(f"Top lower {perc_to_excl*100:.1f}% border:   {high_cut:.6f}")

    hist = (
        ggplot(data_tab.dropna(subset=[col]), aes(x=col))
        + geom_histogram(fill="lightblue", color="black", binwidth=0.1)
        + geom_vline(xintercept=high_cut, color="red", linetype="dashed")
        + geom_vline(xintercept=low_cut, color="blue", linetype="dashed")
    )

    return low_cut, high_cut, hist

In [40]:
import os
path_metadata = os.path.join('resubmission', 'data', 'metadata_selected.csv')
tab_source = pd.read_csv(path_metadata, index_col=0).query(
    'PIR_Nuc_baseline > 10'
    ).dropna(
        subset=['hl_gated_intwise_scaled_logged']
    )
low_cut, high_cut, hist = plot_trimmed_borders(col="hl_gated_intwise_scaled_logged", perc_to_excl=0.5, data_tab=tab_source)
hist

Bottom upper 50.0% border: -1.342584
Top lower 50.0% border:   -0.364742


In [41]:
CONFIG_PATH = 'resubmission/configs/ir_revision_runs.json'
hlrev.set_active_config_path(CONFIG_PATH)
CONFIG = hlrev.load_config()
rows = []
for run_name in sorted(CONFIG["runs"]):
    run = hlrev.resolve_run(run_name)
    low, high = run["class_limits"]
    rows.append({
        "run": run_name,
        "target column": run["class_column"],
        "negative class": f"< {low}",
        "positive class": f">= {high}",
        "results dir": str(run["result_dir"].relative_to(REPO_ROOT)),
        "description": run["description"],
    })
runs_table = pd.DataFrame(rows)
runs_table


,run,target column,negative class,positive class,results dir,description
0,ir_revised,PIR_Nuc_baseline,< 1,>= 30,resubmission/results/models/IR/parnet_512_jc_2...,Base PIR model.


In [43]:
# Main controls
RUN_NAME = "ir_revised" #parnet_512_jc_50percgap_256bp_PIR10
FOLD = "fold4"
CHECKPOINT = "best_model_epoch=27_val_auroc=0.864.ckpt"
REQUIRE_TRAINED = True
REQUIRE_UMAP = True
REQUIRE_MODISCO_INPUTS = True
ATTRIBUTION_EXPORT_MODES = ["final_logit_linearized"]
ATTRIBUTION_MODE = "final_logit_linearized"


In [44]:
RUN_SPEC = hlrev.resolve_run(RUN_NAME)
run_summary = pd.Series({
    "run": RUN_NAME,
    "description": RUN_SPEC["description"],
    "metadata_csv": str(RUN_SPEC["metadata_csv"].relative_to(REPO_ROOT)),
    "target column": RUN_SPEC["class_column"],
    "class limits": tuple(RUN_SPEC["class_limits"]),
    "result_dir": str(RUN_SPEC["result_dir"].relative_to(REPO_ROOT)),
    "training env": str(RUN_SPEC["training_env"]),
    "tfmodisco env": str(RUN_SPEC["modisco_env"]),
    "PIR baseline cutoff": RUN_SPEC.get("pir_nuc_baseline_min"),
    "export attribution modes": ", ".join(ATTRIBUTION_EXPORT_MODES),
    "active attribution mode": ATTRIBUTION_MODE,
})
run_summary


run                                                                ir_revised
description                                                   Base PIR model.
metadata_csv                          resubmission/data/metadata_selected.csv
target column                                                PIR_Nuc_baseline
class limits                                                          (1, 30)
result_dir                  resubmission/results/models/IR/parnet_512_jc_2...
training env                /lustre/groups/crna01/projects/collabs/shared/...
tfmodisco env               /lustre/groups/crna01/projects/collabs/shared/...
PIR baseline cutoff                                                         0
export attribution modes                               final_logit_linearized
active attribution mode                                final_logit_linearized
dtype: object

In [47]:
missing = hlrev.validate_run(
    RUN_NAME,
    require_trained=REQUIRE_TRAINED,
    require_umap=REQUIRE_UMAP,
    require_modisco_inputs=REQUIRE_MODISCO_INPUTS,
    cam_modes=ATTRIBUTION_EXPORT_MODES,
)
missing

[OK] metadata :: /ictstr01/groups/crna01/projects/collabs/gabi/resubmission/data/metadata_selected.csv
[OK] fasta :: /ictstr01/groups/crna01/projects/collabs/gabi/resubmission/data/all_vastdb_introns_50fks.fasta
[OK] parnet weights :: /lustre/groups/crna01/projects/collabs/shared/models/parnet/0.5.0_RBPNet-11M.pt
[OK] training env :: /lustre/groups/crna01/projects/collabs/shared/cenvs/parnet_clean
[OK] training python :: /lustre/groups/crna01/projects/collabs/shared/cenvs/parnet_clean/bin/python
[OK] modisco env :: /lustre/groups/crna01/projects/collabs/shared/cenvs/tfmodisco
[OK] modisco python :: /lustre/groups/crna01/projects/collabs/shared/cenvs/tfmodisco/bin/python
[OK] modisco executable :: /lustre/groups/crna01/projects/collabs/shared/cenvs/tfmodisco/bin/modisco
[OK] result directory :: /ictstr01/groups/crna01/projects/collabs/gabi/resubmission/results/models/IR/parnet_512_jc_256bp
[OK] fold 1 directory :: /ictstr01/groups/crna01/projects/collabs/gabi/resubmission/results/models

[]

## parnet_clean actions

Uncomment only the action you want to run. These all belong to the `parnet_clean` environment.


In [ ]:
# Train all folds for the selected run
# hlrev.train_run(RUN_NAME)

# Train a single fold
# hlrev.train_run(RUN_NAME, folds="3")

# Export UMAP for the best checkpoint or for a chosen fold/checkpoint
hlrev.export_umap_for_run(RUN_NAME, fold=FOLD, checkpoint=CHECKPOINT)

# Export the tf-modisco input arrays under interpretation_fixed/
hlrev.export_modisco_inputs_for_run(
    RUN_NAME,
    fold=FOLD,
    checkpoint=CHECKPOINT,
    cam_modes=ATTRIBUTION_EXPORT_MODES,
)

## tf-modisco step

This cell executes the tf-modisco wrapper directly from the notebook. The notebook still runs in `parnet_clean`, but the wrapper launches the motif/report step with the `tfmodisco` environment.


In [ ]:
MOTIF_DB = Path("/lustre/groups/crna01/projects/collabs/gabi/resubmission/data/external/pwms_all_motifs_ids.meme")
MODISCO_SIDES = "left,right"
MODISCO_WINDOW = 256
MODISCO_SIZE = 31
MODISCO_MAX_SEQLETS = 1000
RUN_MODISCO_REPORT = True

hlrev.run_modisco_for_run(
    RUN_NAME,
    motif_db=MOTIF_DB,
    sides=MODISCO_SIDES,
    window=MODISCO_WINDOW,
    size=MODISCO_SIZE,
    max_seqlets=MODISCO_MAX_SEQLETS,
    skip_report=not RUN_MODISCO_REPORT,
    cam_mode=ATTRIBUTION_MODE,
)


# plotting

## Plotting section

These plots use the saved artifacts for the selected run: UMAP embeddings, the best checkpoint plus fold evaluation results, and the tf-modisco HTML report.


In [49]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from tqdm.auto import tqdm

try:
    from lets_plot import *
    LetsPlot.setup_html()
    base_size = 14
    theme_settings = theme(
        axis_text=element_text(size=base_size),
        axis_title=element_text(size=base_size * 1.2),
        legend_title=element_text(size=base_size * 1.2),
        legend_text=element_text(size=base_size * 0.9),
        plot_title=element_text(size=base_size * 1.4),
        axis_ticks_y=element_line(color='black', size=0.5),
        axis_line_y=element_line(color='black', size=0.5)
    )
except Exception:
    pass

PLOT_OUTPUT_DIR = RUN_SPEC["result_dir"] / "plot_exports"
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CAM_TOP_N = 50
CAM_MIN_LENGTH = 300
SAVE_PLOTS = True

if RUN_SPEC["class_column"] == "stability_ohe":
    NEGATIVE_LABEL = "Unstable"
    POSITIVE_LABEL = "Stable"
else:
    NEGATIVE_LABEL = "SL-RIs"
    POSITIVE_LABEL = "LL-RIs"

NEGATIVE_COLOR = "#a76794"
POSITIVE_COLOR = "#2a1f3f"
CLASS_COLORS = {NEGATIVE_LABEL: NEGATIVE_COLOR, POSITIVE_LABEL: POSITIVE_COLOR}


In [ ]:
deps = hlrev.ensure_training_imports()
checkpoint_path = hlrev.choose_checkpoint(RUN_SPEC, fold=FOLD, explicit=CHECKPOINT)
training = hlrev.load_training_data(RUN_SPEC, deps)
source_data = training["source_data"]
source_seqs = training["source_seqs"]
plot_ckpt_fold = checkpoint_path.parent.name
plot_ckpt_fold

In [58]:
def assign_plot_classes(df, value_col, class_limits, negative_label, positive_label):
    low, high = class_limits
    out = df.copy()
    out["plot_class"] = "else"
    out.loc[out[value_col] < low, "plot_class"] = negative_label
    out.loc[out[value_col] >= high, "plot_class"] = positive_label
    return out

def plot_2d_density_contours_continuous(
    x, y, cmap_name="viridis", kde_smooth=1, levels=11, linewidth=2.3,
    base_alpha=1, min_alpha=0.5, alpha_decay="linear", decay_rate=2.0
):
    xy = np.vstack([x, y])
    kde = gaussian_kde(xy)
    kde.set_bandwidth(bw_method=kde.factor * kde_smooth)
    x_margin = (x.max() - x.min()) * 0.1
    y_margin = (y.max() - y.min()) * 0.1
    xmin, xmax = x.min() - x_margin, x.max() + x_margin
    ymin, ymax = y.min() - y_margin, y.max() + y_margin
    xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    min_level = np.percentile(zz, 50)
    levels_vals = np.linspace(min_level, zz.max(), levels)
    if alpha_decay == "linear":
        alphas = np.linspace(base_alpha, min_alpha, len(levels_vals))
    else:
        exp_range = np.linspace(0, 1, len(levels_vals))
        alphas = base_alpha * np.exp(-decay_rate * exp_range)
    cmap = plt.get_cmap(cmap_name)
    for i, lev in enumerate(levels_vals):
        plt.contour(xx, yy, zz, levels=[lev], colors=[cmap(i / max(levels - 1, 1))], linewidths=linewidth, alpha=alphas[::-1][i])

def plot_2dd_scatter(
    tab_data, contour_col, x_name, y_name, contour_col_classes, scat_alpha=0.3,
    dlevels=11, linewidth=2.3, kde_smooth=1.5, fig_size=(6.5, 5.25),
    show_plot=True, file_name=None
):
    plt.figure(figsize=fig_size)
    sns.scatterplot(data=tab_data, x=x_name, y=y_name, edgecolor=None, color="gray", alpha=scat_alpha, s=8, legend=False, rasterized=True)
    tab_data = tab_data[tab_data[contour_col].isin(list(contour_col_classes.keys()))].copy()
    for cls, cls_col in contour_col_classes.items():
        subset = tab_data[tab_data[contour_col] == cls]
        if subset.shape[0] < 3:
            continue
        cls_cmap = LinearSegmentedColormap.from_list(cls, ["white", cls_col], N=dlevels)
        plot_2d_density_contours_continuous(subset[x_name], subset[y_name], cmap_name=cls_cmap, kde_smooth=kde_smooth, levels=dlevels, linewidth=linewidth, alpha_decay="exponent")
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    if file_name:
        plt.savefig(file_name, bbox_inches="tight", dpi=300)
    if show_plot:
        plt.show()
    else:
        plt.close()


In [ ]:
# 1. UMAP with 2D density contours
umap_path = next(path for path in RUN_SPEC["umap_outputs"] if path.exists())
tab_embeds = pd.read_csv(umap_path, index_col=0)
tab_embeds.index.name = "EVENT"
tab_meta_plot = pd.read_csv(RUN_SPEC["metadata_csv"], index_col=0)
tab_meta_plot = hlrev.filter_metadata_by_pir_cutoff(tab_meta_plot, RUN_SPEC, pd_mod=pd)
tab_umap = tab_meta_plot.merge(tab_embeds, left_index=True, right_index=True, how="inner")
tab_umap = assign_plot_classes(tab_umap, RUN_SPEC["class_column"], RUN_SPEC["class_limits"], NEGATIVE_LABEL, POSITIVE_LABEL)
x_name, y_name = RUN_SPEC["umap_columns"]
umap_save = PLOT_OUTPUT_DIR / f"{RUN_SPEC['name']}_umap_density.png" if SAVE_PLOTS else None
plot_2dd_scatter(tab_umap, "plot_class", x_name=x_name, y_name=y_name, contour_col_classes=CLASS_COLORS, file_name=umap_save)


In [13]:
def _head_bundle(head, feats, mask):
    B, C, L = feats.shape
    if mask is None:
        mask = deps["torch"].ones(B, L, device=feats.device, dtype=feats.dtype)
    mask = mask.to(feats.dtype)
    has_attn = hasattr(head, "attention_pooling")
    if has_attn:
        attn_logits = head.attention_pooling(feats).squeeze(1)
        attn = deps["torch"].softmax(attn_logits.masked_fill(mask == 0, -1e9), dim=-1)
        attn = (attn * mask) / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        pooled = deps["torch"].einsum("bcl,bl->bc", feats, attn)
        denom = None
    else:
        attn = None
        denom = mask.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        pooled = (feats * mask.unsqueeze(1)).sum(-1) / denom
    logits = head.gap_classifier(pooled).squeeze(-1)
    w = head.gap_classifier.weight.squeeze(0)
    b = head.gap_classifier.bias.squeeze(0)
    cam_old = deps["torch"].einsum("c,bcl->bl", w, feats) * mask
    cam_signed = (cam_old + b) * mask
    if has_attn:
        cam_contrib = attn * cam_signed
    else:
        cam_contrib = (cam_signed / denom) * mask
    recon = cam_contrib.sum(dim=-1)
    return dict(cam_old=cam_old, cam_signed=cam_signed, attn=attn, cam_contrib=cam_contrib, seq_logit=logits, recon=recon)

def _fusion_alphas(model_core, logit_left, logit_right):
    joint = deps["torch"].stack([logit_left.detach(), logit_right.detach()], dim=1).requires_grad_(True)
    with deps["torch"].enable_grad():
        if getattr(model_core, "use_joint_classifier", False):
            joint_logit = model_core.joint_classifier(joint).squeeze(-1)
        elif getattr(model_core, "use_simple_fusion", False):
            joint_logit = model_core.simple_fusion(joint).squeeze(-1)
        else:
            joint_logit = joint.mean(dim=1)
        grads = deps["torch"].autograd.grad(joint_logit.sum(), joint, create_graph=False, retain_graph=False)[0]
    return joint_logit.detach(), grads.detach()

def cam_bundle_for_sample(lightning_model, left_onehot, left_mask, right_onehot, right_mask, device="cuda"):
    was_train = lightning_model.training
    lightning_model.eval()
    m = lightning_model.model
    with deps["torch"].no_grad():
        left_onehot = left_onehot.to(device)
        right_onehot = right_onehot.to(device)
        left_mask = left_mask.to(device) if left_mask is not None else None
        right_mask = right_mask.to(device) if right_mask is not None else None
        F_left = m.backbone(left_onehot)
        F_right = m.backbone(right_onehot)
        pack_left = _head_bundle(m.left_cam_head, F_left, left_mask)
        pack_right = _head_bundle(m.right_cam_head, F_right, right_mask)
    joint_logit, grads = _fusion_alphas(m, pack_left["seq_logit"], pack_right["seq_logit"])
    pack_left["fusion_alpha"] = grads[:, 0]
    pack_right["fusion_alpha"] = grads[:, 1]
    pack_left["cam_branch_contrib"] = pack_left["cam_contrib"]
    pack_right["cam_branch_contrib"] = pack_right["cam_contrib"]
    pack_left["cam_final_contrib"] = pack_left["cam_contrib"] * pack_left["fusion_alpha"].unsqueeze(1)
    pack_right["cam_final_contrib"] = pack_right["fusion_alpha"].unsqueeze(1) * pack_right["cam_contrib"]
    if was_train:
        lightning_model.train()
    return {"left": pack_left, "right": pack_right, "joint_logit": joint_logit}


def cam_values_for_mode(pack, attribution_mode):
    if hlrev.normalize_attribution_mode(attribution_mode) == hlrev.DEFAULT_ATTRIBUTION_MODE:
        return pack["cam_final_contrib"]
    return pack["cam_branch_contrib"]


In [ ]:
# 2. CAM heatmap
best_model = hlrev.load_best_model(RUN_SPEC, deps, checkpoint_path)
device = "cuda" if deps["torch"].cuda.is_available() else "cpu"
eval_path = checkpoint_path.parent / "evaluation_results.csv"
res_val = pd.read_csv(eval_path)
res_val = res_val[res_val["label"] == res_val["binary_prediction"]].copy()
length_df = pd.DataFrame({"id": list(source_seqs.keys()), "length": [len(seq) for seq in source_seqs.values()]})
res_val = res_val.merge(length_df, on="id", how="left")
res_val = res_val[res_val["length"] >= CAM_MIN_LENGTH].copy()
top_pos = res_val[res_val.label == 1].sort_values(by="prediction", ascending=False).head(CAM_TOP_N).copy()
top_neg = res_val[res_val.label == 0].sort_values(by="prediction", ascending=True).head(CAM_TOP_N).copy()
binder_grps = []
class_to_df = {POSITIVE_LABEL: top_pos, NEGATIVE_LABEL: top_neg}
for grp_name, grp_df in class_to_df.items():
    grp_ids = grp_df.id.tolist()
    ds_grp = deps["irt"].IntronsEndsDataset(grp_ids, source_seqs, source_data, L_left=RUN_SPEC["flank_length"], L_right=RUN_SPEC["flank_length"], use_middle=False)
    binder_seqs = []
    for seq_id in tqdm(grp_ids, total=len(grp_ids)):
        packs = cam_bundle_for_sample(best_model, ds_grp[seq_id]["left_onehot"].unsqueeze(0), ds_grp[seq_id]["left_mask"].unsqueeze(0), ds_grp[seq_id]["right_onehot"].unsqueeze(0), ds_grp[seq_id]["right_mask"].unsqueeze(0), device=device)
        binder_cams = []
        for side, cam_key in [("left", "cam_left"), ("right", "cam_right")]:
            cam_contrib = cam_values_for_mode(packs[side], ATTRIBUTION_MODE).squeeze(0).detach().cpu().numpy()
            binder_cams.append(pd.DataFrame({"id": seq_id, "class": grp_name, "cam_head": f"{cam_key}_contrib", "position": np.arange(1, cam_contrib.size + 1), "cam_scores": cam_contrib}))
        binder_seqs.append(pd.concat(binder_cams, axis=0))
    binder_grps.append(pd.concat(binder_seqs, axis=0))
tab_topcam = pd.concat(binder_grps, axis=0)
tab_topcam["cam_scores"] = tab_topcam["cam_scores"].clip(-0.1, 0.1)
cam_plot = (
    ggplot(data=tab_topcam)
    + geom_tile(aes(x="position", y="id", fill="cam_scores"), tooltips="none", width=1)
    + scale_fill_gradient2(low=NEGATIVE_COLOR, mid="white", high=POSITIVE_COLOR, midpoint=0)
    + facet_grid(x="cam_head", y="class", scales="free")
    + labs(x="Position at intron end", y="Introns", fill="CAM score")
    + theme_settings
    + theme(axis_text_y=element_blank())
    + ggsize(1120, 400)
)
cam_plot


In [59]:
if SAVE_PLOTS:
    ggsave(
        cam_plot,
        path=str(PLOT_OUTPUT_DIR),
        filename=f"{RUN_SPEC['name']}_CAMprofiles{hlrev.attribution_mode_suffix(ATTRIBUTION_MODE)}.svg",
    )

In [52]:
# 3. Significant tf-modisco hits
speck_candidates = [
    REPO_ROOT / "resubmission/data/external/ENCORE/HepG2-HeLa_FeaturesMatrix.csv",
    REPO_ROOT / "data/speckle_rbps/HepG2-HeLa_FeaturesMatrix.csv",
]
speck_path = next((path for path in speck_candidates if path.exists()), None)
speck_rbps = []
if speck_path is not None:
    speck_tab = pd.read_csv(speck_path)
    if "Speckles" in speck_tab.columns and "RBP" in speck_tab.columns:
        speck_rbps = speck_tab.dropna(subset=["Speckles"])["RBP"].tolist()

modisco_tables = []
modisco_prefix = hlrev.attribution_mode_prefix(RUN_SPEC["modisco_prefix"], ATTRIBUTION_MODE)
for side in ["left", "right"]:
    motif_html = RUN_SPEC["interpretation_dir"] / f"{modisco_prefix}_{side}_modisco" / "report" / "motifs.html"
    dfs = pd.read_html(motif_html, flavor="lxml")
    binder = []
    for i in range(0, 3):
        tmp = dfs[0][["pattern", "num_seqlets", f"match{i}", f"qval{i}"]].copy()
        tmp = tmp.rename(columns={f"match{i}": "match", f"qval{i}": "qval"})
        tmp["match_n"] = str(i + 1)
        binder.append(tmp)
    side_df = pd.concat(binder, axis=0)
    side_df["qval_log"] = -np.log10(side_df["qval"])
    side_df["match_name"] = side_df["match"].astype(str).str.split("_").str[0]
    side_df["cam_side"] = side
    modisco_tables.append(side_df)
dfs_bpes = pd.concat(modisco_tables, axis=0)
dfs_bpes["significant"] = dfs_bpes["qval"] < 0.05
dfs_bpes["label"] = np.where(dfs_bpes["significant"], dfs_bpes["match_name"], np.nan)
dfs_bpes["significance"] = np.where(dfs_bpes["significant"], "Significant", "NS")
dfs_bpes["pattern_name"] = dfs_bpes["pattern"].astype(str).str.split("_").str[0]
dfs_bpes["in_speckle"] = dfs_bpes["match_name"].isin(speck_rbps).map({True: "yes", False: "no"})
hits_plot = (
    ggplot(data=dfs_bpes)
    + geom_hline(yintercept=-np.log10(0.05), linetype="dashed", color="red")
    + geom_point(aes(x="num_seqlets", y="qval_log", fill="significance", alpha="significance"), shape=21)
    + scale_fill_manual(values={"NS": "grey", "Significant": "red"})
    + scale_alpha_manual(values={"NS": 0.1, "Significant": 1})
    + geom_text_repel(aes(x="num_seqlets", y="qval_log", label="label", color="in_speckle"), size=6, na_text="")
    + scale_color_manual(values={"yes": "red", "no": "black"})
    + coord_cartesian(xlim=[0, 100], ylim=[-0.2, 4])
    + facet_grid(y="pattern_name")
    + theme_settings
    + ggsize(550, 700)
)
hits_plot

In [54]:
if SAVE_PLOTS:
    ggsave(
        hits_plot,
        path=str(PLOT_OUTPUT_DIR),
        filename=f"{RUN_SPEC['name']}_modisco_hits{hlrev.attribution_mode_suffix(ATTRIBUTION_MODE)}.svg",
    )
    dfs_bpes[['pattern', 'match_name', 'pattern_name', 'cam_side', 'num_seqlets', 'qval', 'qval_log']].dropna().to_csv(
        PLOT_OUTPUT_DIR / f"{RUN_SPEC['name']}_modisco_hits{hlrev.attribution_mode_suffix(ATTRIBUTION_MODE)}.csv",
        index=False
    )

## Fold performance of selected run

This reproduces the per-fold ROC/PRC summary style for the currently selected run.


In [56]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

def compute_roc_prc(y_true, y_pred):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    return {
        "roc": {"fpr": fpr, "tpr": tpr},
        "prc": {"precision": precision, "recall": recall},
    }

val_metrics = []
val_curves_prc = []
val_curves_roc = []
model_label = RUN_SPEC["name"]

for fld_dir in sorted(RUN_SPEC["result_dir"].glob("beststate_fold*")):
    eval_file = fld_dir / "evaluation_results.csv"
    if not eval_file.exists():
        continue
    res_val_fold = pd.read_csv(eval_file)
    val_auc = roc_auc_score(res_val_fold["label"], res_val_fold["prediction"])
    val_ap = average_precision_score(res_val_fold["label"], res_val_fold["prediction"])
    fold_name = fld_dir.name.replace("beststate_fold", "")
    val_metrics.append({"model": model_label, "fold": fold_name, "ROC": val_auc, "PRC": val_ap})
    curves = compute_roc_prc(res_val_fold["label"], res_val_fold["prediction"])
    val_curves_roc.append(pd.DataFrame({"model": model_label, "fold": fold_name, "tpr": curves["roc"]["tpr"], "fpr": curves["roc"]["fpr"]}))
    val_curves_prc.append(pd.DataFrame({"model": model_label, "fold": fold_name, "precision": curves["prc"]["precision"], "recall": curves["prc"]["recall"]}))

tab_metrics = pd.DataFrame(val_metrics).melt(value_vars=["ROC", "PRC"], value_name="AU", var_name="metrics", id_vars=["model", "fold"])
tab_curves_roc = pd.concat(val_curves_roc, axis=0)
tab_curves_prc = pd.concat(val_curves_prc, axis=0)
tab_metrics


,model,fold,metrics,AU
0,ir_revised,1,ROC,0.863050
1,ir_revised,2,ROC,0.856238
2,ir_revised,3,ROC,0.862108
3,ir_revised,4,ROC,0.864444
4,ir_revised,5,ROC,0.853523
5,ir_revised,1,PRC,0.827476
6,ir_revised,2,PRC,0.818284
7,ir_revised,3,PRC,0.819171
8,ir_revised,4,PRC,0.825639
9,ir_revised,5,PRC,0.811975


In [57]:
tab_curves_prc_plt = tab_curves_prc.query("precision > 0").copy()
pos_frac = tab_curves_prc_plt["precision"].min()
best_prc_row = max(val_metrics, key=lambda row: row["PRC"])
best_roc_row = max(val_metrics, key=lambda row: row["ROC"])
prc_annot = pd.DataFrame({
    "x": [0.98],
    "y": [0.98],
    "label": [f"Best fold {best_prc_row['fold']}: AUPRC={best_prc_row['PRC']:.3f}"],
})
roc_annot = pd.DataFrame({
    "x": [0.98],
    "y": [0.04],
    "label": [f"Best fold {best_roc_row['fold']}: AUROC={best_roc_row['ROC']:.3f}"],
})

perf_prc = (
    ggplot(tab_curves_prc_plt)
    + geom_line(aes(group="fold", y="precision", x="recall"), color=POSITIVE_COLOR, size=0.8, alpha=0.7, tooltips="none")
    + geom_hline(yintercept=pos_frac, linetype="dashed", color=POSITIVE_COLOR, alpha=0.7, size=1)
    + geom_text(aes(x="x", y="y", label="label"), data=prc_annot, color=POSITIVE_COLOR, hjust=1, vjust=1)
    + theme_settings
    + labs(x="Recall", y="Precision", title=f"{RUN_SPEC['name']} PRC across folds")
    + coord_cartesian(xlim=[0, 1], ylim=[0, 1])
    + theme_settings
    + ggsize(550, 450)
)
perf_prc

perf_roc = (
    ggplot(tab_curves_roc)
    + geom_line(aes(group="fold", y="tpr", x="fpr"), color=POSITIVE_COLOR, size=0.8, alpha=0.7)
    + geom_abline(intercept=0, slope=1, linetype="dashed", size=1)
    + geom_text(aes(x="x", y="y", label="label"), data=roc_annot, color=POSITIVE_COLOR, hjust=1, vjust=0)
    + theme_settings
    + labs(y="True Positive Rate", x="False Positive Rate", title=f"{RUN_SPEC['name']} ROC across folds")
    + coord_cartesian(xlim=[0, 1], ylim=[0, 1])
    + theme_settings
    + ggsize(550, 450)
)
perf_roc

if SAVE_PLOTS:
    ggsave(perf_prc, filename=f"{RUN_SPEC['name']}_auprc_folds.svg", path=str(PLOT_OUTPUT_DIR))
    ggsave(perf_roc, filename=f"{RUN_SPEC['name']}_auroc_folds.svg", path=str(PLOT_OUTPUT_DIR))
    tab_metrics.to_csv(PLOT_OUTPUT_DIR / f"{RUN_SPEC['name']}_performance_metrics.csv", index=False)
    tab_curves_prc.to_csv(PLOT_OUTPUT_DIR / f"{RUN_SPEC['name']}_prc_curves.csv", index=False)
    tab_curves_roc.to_csv(PLOT_OUTPUT_DIR / f"{RUN_SPEC['name']}_roc_curves.csv", index=False)
